# Reimbursement claims: supervised multi-agent PoC

Run this notebook top to bottom. All records, plans and IDs below are synthetic. This demo is offline by default: it does not call an LLM, parse files, or connect to an insurer. The editable implementation lives in `src/reimbursement_poc/`.

**Decision boundary:** rules decide; ambiguity and inconsistency go to a person. A risk signal is not a fraud finding.

## 1. Load the policy catalogue

The catalogue is intentionally tiny and fictional. A production design would retrieve versioned plan terms from an approved source.

In [ ]:
from reimbursement_poc.domain import ClaimRequest
from reimbursement_poc.observability import configure_logging
from reimbursement_poc.orchestrator import CaseOrchestrator

configure_logging()

POLICIES = {
    "PPO-100": {
        "active": True,
        "covered_procedures": ["MRI-KNEE", "CARDIO-CONSULT"],
        "max_reimbursement": 750,
    },
    "BASIC-200": {
        "active": True,
        "covered_procedures": ["CARDIO-CONSULT"],
        "max_reimbursement": 250,
    },
}
orchestrator = CaseOrchestrator(POLICIES)

## 2. Submit synthetic examples

The payload represents fields that a real intake/OCR system would extract. This PoC validates the workflow after extraction; it does not claim to validate original files.

In [ ]:
def make_claim(
    case_id,
    plan_id="PPO-100",
    procedure="MRI-KNEE",
    invoice_id=None,
    invoice_date="2026-09-01",
    request_date="2026-09-01",
    include_physician=True,
):
    medical_request = {"procedure_code": procedure, "service_date": request_date}
    if include_physician:
        medical_request["physician_id"] = "DR-SYNTHETIC"
    return ClaimRequest(
        case_id=case_id,
        member_id="MEMBER-SYNTHETIC",
        plan_id=plan_id,
        documents={
            "invoice": {
                "invoice_id": invoice_id or f"INV-{case_id}",
                "amount": 900,
                "provider": "SYNTHETIC CLINIC",
                "service_date": invoice_date,
                "procedure_code": procedure,
            },
            "medical_request": medical_request,
        },
    )


examples = [
    make_claim("CASE-1001"),
    make_claim("CASE-1002", include_physician=False),
    make_claim("CASE-1003", plan_id="BASIC-200"),
    make_claim("CASE-1004", invoice_id="INV-CASE-1001", invoice_date="2026-09-02"),
]

results = [orchestrator.submit(request) for request in examples]
for result in results:
    print(
        {
            "case_id": result.request.case_id,
            "status": result.status.value,
            "reasons": result.reasons,
            "missing_items": result.missing_items,
            "risk_flags": result.risk_flags,
            "approved_amount": result.approved_amount,
        }
    )

## 3. Customer status and local observability

The support response is intentionally member-safe; it does not disclose internal risk signals. Telemetry has only bounded stage/outcome labels and timings, not member IDs or document contents.

In [ ]:
print(orchestrator.customer_update("CASE-1002"))
print(orchestrator.customer_update("CASE-1004"))
orchestrator.observability.snapshot()

## Optional: LLM-assisted wording (explicit opt-in)

This cell is deliberately not run automatically. After reviewing provider approval and data-handling requirements, load your local `.env` and call the adapter with generic reason codes only. Never pass real health, financial, member or claim data.

In [ ]:
# from dotenv import load_dotenv
# load_dotenv()
# from reimbursement_poc.llm import explain_decision
# explain_decision("NEEDS_INFORMATION", ("required_document_fields_missing",))